# 01 — Exploratory Data Analysis
## Taiwan Credit Card Default Dataset

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

## 1. Load Data

In [ ]:
from src.data.download import download_dataset
from src.data.preprocess import clean_dataframe, CONTINUOUS_FEATURES, CATEGORICAL_FEATURES, ORDINAL_FEATURES, TARGET_COL

df_raw = download_dataset()
df = clean_dataframe(df_raw)
print(f'Shape: {df.shape}')
df.head()

## 2. Basic Statistics

In [ ]:
print('=== Target Distribution ===')
print(df[TARGET_COL].value_counts(normalize=True))
print(f'\nDefault rate: {df[TARGET_COL].mean():.3f}')

print('\n=== Continuous Features ===')
df[CONTINUOUS_FEATURES].describe().round(2)

In [ ]:
print('=== Categorical Features ===')
for col in CATEGORICAL_FEATURES:
    print(f'\n{col}:')
    print(df[col].value_counts().sort_index())

print('\n=== Ordinal Features (Repayment Status) ===')
for col in ORDINAL_FEATURES:
    print(f'\n{col}:')
    print(df[col].value_counts().sort_index())

## 3. Distributions

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
for i, col in enumerate(CONTINUOUS_FEATURES):
    ax = axes[i // 5, i % 5]
    df[col].hist(bins=50, ax=ax, alpha=0.7)
    ax.set_title(col)
for j in range(len(CONTINUOUS_FEATURES), 15):
    axes[j // 5, j % 5].set_visible(False)
plt.tight_layout()
plt.suptitle('Continuous Feature Distributions', y=1.02, fontsize=14)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, col in enumerate(CATEGORICAL_FEATURES):
    df.groupby(col)[TARGET_COL].mean().plot(kind='bar', ax=axes[i], color='steelblue')
    axes[i].set_title(f'Default Rate by {col}')
    axes[i].set_ylabel('Default Rate')
plt.tight_layout()
plt.show()

## 4. Correlation Analysis

In [ ]:
corr = df.corr()

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, annot=False, cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

print('\nCorrelations with target:')
print(corr[TARGET_COL].drop(TARGET_COL).sort_values(ascending=False).round(3))

## 5. Default Rate by Repayment Status

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, col in enumerate(ORDINAL_FEATURES):
    ax = axes[i // 3, i % 3]
    rates = df.groupby(col)[TARGET_COL].agg(['mean', 'count'])
    rates['mean'].plot(kind='bar', ax=ax, color='coral')
    ax.set_title(f'Default Rate by {col}')
    ax.set_ylabel('Default Rate')
    # Add count labels
    for j, (_, row) in enumerate(rates.iterrows()):
        ax.text(j, row['mean'] + 0.01, f'n={int(row["count"])}', ha='center', fontsize=7)
plt.tight_layout()
plt.show()

## 6. Key Observations

Document findings here after running the notebook:
- Class imbalance: ~22% default rate
- PAY_1 (most recent repayment status) is the strongest predictor
- BILL_AMT features are highly correlated with each other
- LIMIT_BAL shows inverse relationship with default